| Coluna | Descrição / cálculo |
|---|---|
| `estado` | Nome do estado onde está localizado o município. |
| `uf` | Sigla da Unidade Federativa, como `PB`, `SC`, `SP`. |
| `municipio` | Nome do município onde os votos foram contabilizados. |
| `codigo_tse` | Código utilizado pelo TSE para identificar o município. Também é utilizado para montar a URL de consulta dos resultados. |
| `codigo_ibge` | Código do município segundo o IBGE. Pode ser utilizado para cruzar os dados eleitorais com mapas e indicadores socioeconômicos. |
| `numero_candidato` | Número eleitoral do candidato à Presidência. |
| `nome_candidato` | Nome completo do candidato registrado no TSE. |
| `nome_urna` | Nome do candidato apresentado ao eleitor na urna eletrônica. |
| `partido` | Sigla do partido político do candidato. |
| `destinacao_voto` | Indica como os votos recebidos pelo candidato serão considerados na totalização. |
| `situacao` | Situação do candidato no processo eleitoral, conforme informação do TSE. |
| `eleito` | Indicador fornecido pelo TSE sobre a situação eleitoral do candidato. |
| `votos_candidato` | Quantidade de votos recebidos pelo candidato naquele município. |
| `percentual_candidato` | Percentual de votos obtidos pelo candidato no município. |
| `percentual_candidato_preciso` | Versão mais precisa do percentual de votos do candidato, quando disponível no JSON. |
| `total_votos` | Total de votos contabilizados no município para o cargo de Presidente. |
| `votos_validos` | Total de votos considerados válidos para os candidatos. |
| `votos_brancos` | Quantidade de votos em branco registrados no município. |
| `total_votos_nulos` | Total de votos nulos. Corresponde aos votos nulos comuns somados aos nulos técnicos. |
| `votos_nulos` | Quantidade de votos nulos comuns. |
| `votos_nulos_tecnicos` | Quantidade de votos classificados tecnicamente como nulos pelo sistema de totalização. |
| `votos_anulados` | Quantidade de votos anulados conforme as regras da Justiça Eleitoral. |
| `votos_anulados_sub_judice` | Quantidade de votos anulados relacionados a situações que ainda dependem de decisão judicial. |
| `votos_invalidos` | Coluna criada no código. Cálculo: `total_votos_nulos + votos_anulados + votos_anulados_sub_judice`. |
| `votos_nao_validos` | Coluna criada no código. Cálculo: `votos_brancos + votos_invalidos`. |
| `votos_sem_candidato` | Quantidade de votos registrados sem candidato correspondente, quando essa situação existir. |
| `votos_secoes_anuladas` | Quantidade de votos relacionados a seções eleitorais anuladas, quando aplicável. |
| `eleitorado` | Quantidade total de eleitores aptos a votar naquela abrangência. |
| `comparecimento` | Quantidade de eleitores que compareceram para votar. |
| `abstencoes` | Quantidade de eleitores que não compareceram. Conceitualmente: `eleitorado - comparecimento`. |
| `total_secoes` | Quantidade total de seções eleitorais existentes no município. |
| `secoes_totalizadas` | Quantidade de seções que já tiveram seus resultados incorporados à totalização. |
| `secoes_nao_totalizadas` | Quantidade de seções que ainda não foram incorporadas à totalização. |
| `percentual_secoes_totalizadas` | Percentual das seções eleitorais que já foram totalizadas. |
| `andamento` | Indica o estágio atual da totalização dos votos. |
| `totalizacao_final` | Indica se a totalização daquele resultado já foi finalizada. |
| `data_totalizacao` | Data relacionada à totalização dos resultados. |
| `hora_totalizacao` | Horário relacionado à totalização dos resultados. |
| `data_geracao` | Data em que o arquivo JSON foi gerado pelo sistema do TSE. |
| `hora_geracao` | Horário em que o arquivo JSON foi gerado pelo sistema do TSE. |

In [ ]:
import requests
import pandas as pd
import time

# ============================================================
# CONFIGURAÇÕES
# ============================================================

url_base = "https://resultados.tse.jus.br"

ambiente = "oficial"

ciclo = "ele2026" # para 2022 "ele2022" (1º turno em 2022 codigo_eleicao = 544 | 2º turno em 2022 codigo_eleicao = 545)

codigo_eleicao = 6257 # para segundo turno, ver o código que será disponibilizado pelo o TSE

codigo_eleicao_formatado = f"{codigo_eleicao:06d}"

codigo_cargo = "0001"  # Presidente

"""
| Cargo              | Código |

| Presidente         | `0001` |
| Governador         | `0003` |
| Senador            | `0005` |
| Deputado Federal   | `0006` |
| Deputado Estadual  | `0007` |
| Deputado Distrital | `0008` |
| Prefeito           | `0011` |
| Vereador           | `0013` |
"""

# ============================================================
# 1. BUSCAR LISTA OFICIAL DE MUNICÍPIOS
# ============================================================

url_municipios = (
    f"{url_base}/"
    f"{ambiente}/"
    f"{ciclo}/"
    f"{codigo_eleicao}/"
    f"config/"
    f"mun-e{codigo_eleicao_formatado}-cm.json"
)


resposta = requests.get(
    url_municipios,
    timeout=30
)

resposta.raise_for_status()

dados_municipios = resposta.json()


# ============================================================
# 2. CRIAR LISTA COM TODOS OS MUNICÍPIOS
# ============================================================

municipios = []

for estado in dados_municipios["abr"]:

    uf = estado["cd"].lower()

    nome_estado = estado["ds"]

    # Ignora localidades do exterior
    if uf == "zz":
        continue

    for municipio in estado["mu"]:

        codigo_tse = str(
            municipio["cd"]
        ).zfill(5)

        codigo_ibge = str(
            municipio.get("cdi", "")
        ).zfill(5)

        nome_municipio = municipio["nm"]

        municipios.append({

            "uf": uf,

            "estado": nome_estado,

            "codigo_tse": codigo_tse,

            "codigo_ibge": codigo_ibge,

            "municipio": nome_municipio

        })


print(
    "Quantidade de municípios encontrados:",
    len(municipios)
)


# ============================================================
# 3. LISTA QUE RECEBERÁ TODOS OS RESULTADOS
# ============================================================

resultados = []


# ============================================================
# 4. PERCORRER TODOS OS MUNICÍPIOS DO BRASIL
# ============================================================

for indice, municipio in enumerate(municipios, start=1):

    uf = municipio["uf"]

    estado = municipio["estado"]

    codigo_tse = municipio["codigo_tse"]

    codigo_ibge = municipio["codigo_ibge"]

    nome_municipio = municipio["municipio"]


    # --------------------------------------------------------
    # URL DO RESULTADO PRESIDENCIAL DO MUNICÍPIO
    # --------------------------------------------------------

    url_resultado = (
        f"{url_base}/"
        f"{ambiente}/"
        f"{ciclo}/"
        f"{codigo_eleicao}/"
        f"dados/{uf}/"
        f"{uf}{codigo_tse}"
        f"-c{codigo_cargo}"
        f"-e{codigo_eleicao_formatado}"
        f"-u.json"
    )


    try:

        resposta = requests.get(
            url_resultado,
            timeout=30
        )


        # Caso o arquivo ainda não esteja disponível
        if resposta.status_code == 404:

            print(
                f"Arquivo não encontrado: "
                f"{nome_municipio} - {uf.upper()}"
            )

            continue


        resposta.raise_for_status()

        dados = resposta.json()


        # ====================================================
        # 5. DADOS GERAIS DA VOTAÇÃO DO MUNICÍPIO
        # ====================================================

        votos = dados.get(
            "v",
            {}
        )

        eleitores = dados.get(
            "e",
            {}
        )

        secoes = dados.get(
            "s",
            {}
        )


        # ----------------------------------------------------
        # VOTOS
        # ----------------------------------------------------

        total_votos = votos.get(
            "tv",
            0
        )

        votos_validos = votos.get(
            "vv",
            0
        )

        votos_brancos = votos.get(
            "vb",
            0
        )

        total_nulos = votos.get(
            "tvn",
            0
        )

        votos_nulos = votos.get(
            "vn",
            0
        )

        votos_nulos_tecnicos = votos.get(
            "vnt",
            0
        )

        votos_anulados = votos.get(
            "van",
            0
        )

        votos_anulados_sub_judice = votos.get(
            "vansj",
            0
        )

        votos_sem_candidato = votos.get(
            "vscv",
            0
        )

        votos_secoes_anuladas = votos.get(
            "vsan",
            0
        )


        # ----------------------------------------------------
        # CRIAMOS UMA COLUNA ANALÍTICA DE "INVÁLIDOS"
        #
        # Não existe no TSE um campo chamado "invalidos".
        #
        # Aqui estamos considerando:
        #
        # nulos
        # + anulados
        # + anulados sub judice
        #
        # O total_nulos já contém:
        # votos_nulos + votos_nulos_tecnicos
        # ----------------------------------------------------

        votos_invalidos = (
            total_nulos
            + votos_anulados
            + votos_anulados_sub_judice
        )


        # ----------------------------------------------------
        # BRANCOS + INVÁLIDOS
        # ----------------------------------------------------

        votos_nao_validos = (
            votos_brancos
            + votos_invalidos
        )


        # ====================================================
        # 6. DADOS DOS ELEITORES
        # ====================================================

        eleitorado = eleitores.get(
            "te",
            0
        )

        comparecimento = eleitores.get(
            "c",
            0
        )

        abstencoes = eleitores.get(
            "a",
            0
        )


        # ====================================================
        # 7. DADOS DAS SEÇÕES
        # ====================================================

        total_secoes = secoes.get(
            "ts",
            0
        )

        secoes_totalizadas = secoes.get(
            "st",
            0
        )

        secoes_nao_totalizadas = secoes.get(
            "snt",
            0
        )

        percentual_secoes_totalizadas = secoes.get(
            "pstn",
            0
        )


        # ====================================================
        # 8. PERCORRER OS CANDIDATOS
        # ====================================================

        for cargo in dados.get("carg", []):

            for agrupamento in cargo.get("agr", []):

                for partido in agrupamento.get("par", []):

                    for candidato in partido.get("cand", []):


                        # ====================================
                        # 9. UMA LINHA DO DATAFRAME
                        # ====================================

                        resultados.append({

                            # -------------------------------
                            # LOCALIZAÇÃO
                            # -------------------------------

                            "estado": estado,

                            "uf": uf.upper(),

                            "municipio": nome_municipio,

                            "codigo_tse": codigo_tse,

                            "codigo_ibge": codigo_ibge,


                            # -------------------------------
                            # CANDIDATO
                            # -------------------------------

                            "numero_candidato":
                                candidato.get("n"),

                            "nome_candidato":
                                candidato.get("nm"),

                            "nome_urna":
                                candidato.get("nmu"),

                            "partido":
                                partido.get("sg"),

                            "destinacao_voto":
                                candidato.get("dvt"),

                            "situacao":
                                candidato.get("st"),

                            "eleito":
                                candidato.get("e"),


                            # -------------------------------
                            # VOTAÇÃO DO CANDIDATO
                            # -------------------------------

                            "votos_candidato":
                                candidato.get("vap", 0),

                            "percentual_candidato":
                                candidato.get("pvap"),

                            "percentual_candidato_preciso":
                                candidato.get("pvapn"),


                            # -------------------------------
                            # TOTALIZAÇÃO DO MUNICÍPIO
                            # -------------------------------

                            "total_votos":
                                total_votos,

                            "votos_validos":
                                votos_validos,

                            "votos_brancos":
                                votos_brancos,

                            "total_votos_nulos":
                                total_nulos,

                            "votos_nulos":
                                votos_nulos,

                            "votos_nulos_tecnicos":
                                votos_nulos_tecnicos,

                            "votos_anulados":
                                votos_anulados,

                            "votos_anulados_sub_judice":
                                votos_anulados_sub_judice,

                            "votos_invalidos":
                                votos_invalidos,

                            "votos_nao_validos":
                                votos_nao_validos,

                            "votos_sem_candidato":
                                votos_sem_candidato,

                            "votos_secoes_anuladas":
                                votos_secoes_anuladas,


                            # -------------------------------
                            # ELEITORADO
                            # -------------------------------

                            "eleitorado":
                                eleitorado,

                            "comparecimento":
                                comparecimento,

                            "abstencoes":
                                abstencoes,


                            # -------------------------------
                            # SEÇÕES
                            # -------------------------------

                            "total_secoes":
                                total_secoes,

                            "secoes_totalizadas":
                                secoes_totalizadas,

                            "secoes_nao_totalizadas":
                                secoes_nao_totalizadas,

                            "percentual_secoes_totalizadas":
                                percentual_secoes_totalizadas,


                            # -------------------------------
                            # CONTROLE DA TOTALIZAÇÃO
                            # -------------------------------

                            "andamento":
                                dados.get("and"),

                            "totalizacao_final":
                                dados.get("tf"),

                            "data_totalizacao":
                                dados.get("dt"),

                            "hora_totalizacao":
                                dados.get("ht"),

                            "data_geracao":
                                dados.get("dg"),

                            "hora_geracao":
                                dados.get("hg")

                        })


        # ====================================================
        # MOSTRAR PROGRESSO
        # ====================================================

        if indice % 100 == 0:

            print(
                f"{indice} de "
                f"{len(municipios)} "
                f"municípios processados"
            )


        # ====================================================
        # PEQUENA PAUSA ENTRE REQUISIÇÕES
        # ====================================================

        time.sleep(0.03)


    except requests.RequestException as erro:

        print(
            f"Erro em "
            f"{nome_municipio} - "
            f"{uf.upper()}: "
            f"{erro}"
        )


# ============================================================
# 10. CRIAR UM ÚNICO DATAFRAME
# ============================================================

df_eleicao = pd.DataFrame(
    resultados
)


# ============================================================
# 11. ORGANIZAR O DATAFRAME
# ============================================================

df_eleicao = df_eleicao.sort_values(
    by=[
        "uf",
        "municipio",
        "votos_candidato"
    ],
    ascending=[
        True,
        True,
        False
    ]
).reset_index(
    drop=True
)


# ============================================================
# 12. RESULTADO
# ============================================================

print()

print(
    "Quantidade de linhas:",
    len(df_eleicao)
)

print()

print(
    "Quantidade de municípios:",
    df_eleicao["municipio"].nunique()
)

print()

print(
    "Quantidade de candidatos:",
    df_eleicao["nome_urna"].nunique()
)


df_eleicao

Quantidade de municípios encontrados: 5571
100 de 5571 municípios processados
200 de 5571 municípios processados
300 de 5571 municípios processados
400 de 5571 municípios processados
500 de 5571 municípios processados
600 de 5571 municípios processados
700 de 5571 municípios processados
800 de 5571 municípios processados
900 de 5571 municípios processados
1000 de 5571 municípios processados
1100 de 5571 municípios processados
1200 de 5571 municípios processados
1300 de 5571 municípios processados
1400 de 5571 municípios processados
1500 de 5571 municípios processados
1600 de 5571 municípios processados
1700 de 5571 municípios processados
1800 de 5571 municípios processados
1900 de 5571 municípios processados
2000 de 5571 municípios processados
2100 de 5571 municípios processados
2200 de 5571 municípios processados
2300 de 5571 municípios processados
2400 de 5571 municípios processados
2500 de 5571 municípios processados
2600 de 5571 municípios processados
2700 de 5571 municípios proces

,estado,uf,municipio,codigo_tse,codigo_ibge,numero_candidato,nome_candidato,nome_urna,partido,destinacao_voto,...,total_secoes,secoes_totalizadas,secoes_nao_totalizadas,percentual_secoes_totalizadas,andamento,totalizacao_final,data_totalizacao,hora_totalizacao,data_geracao,hora_geracao
0,ACRE,AC,ACRELÂNDIA,01120,1200013,14,RENAN ANTONIO FERREIRA DOS SANTOS,RENAN SANTOS,MISSÃO,Válido,...,45,45,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:13
1,ACRE,AC,ACRELÂNDIA,01120,1200013,22,FLAVIO NANTES BOLSONARO,FLAVIO BOLSONARO,PL,Válido,...,45,45,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:13
2,ACRE,AC,ACRELÂNDIA,01120,1200013,30,ROMEU ZEMA NETO,ZEMA,NOVO,Válido,...,45,45,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:13
3,ACRE,AC,ACRELÂNDIA,01120,1200013,27,CLARIANA ZACARKIM BARAO,CLARIANA BARAO,DC,Válido,...,45,45,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:13
4,ACRE,AC,ACRELÂNDIA,01120,1200013,80,SAMARA MARTINS DA SILVA FEITOSA,SAMARA,UP,Válido,...,45,45,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66847,TOCANTINS,TO,XAMBIOÁ,96431,1722107,21,EDMILSON SILVA COSTA,EDMILSON COSTA,PCB,Válido,...,35,35,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:17
66848,TOCANTINS,TO,XAMBIOÁ,96431,1722107,27,CLARIANA ZACARKIM BARAO,CLARIANA BARAO,DC,Válido,...,35,35,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:17
66849,TOCANTINS,TO,XAMBIOÁ,96431,1722107,35,WILSON GRASSI JUNIOR,VETERINÁRIO WILSON GRASSI,DEMOCRATA,Válido,...,35,35,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:17
66850,TOCANTINS,TO,XAMBIOÁ,96431,1722107,80,SAMARA MARTINS DA SILVA FEITOSA,SAMARA,UP,Válido,...,35,35,0,100,f,s,05/10/2026,12:51:05,05/10/2026,12:52:17


In [ ]:
 "https://resultados.tse.jus.br"

ambiente = "oficial"

ciclo = "ele2026" # para 2022 "ele2022" (1º turno em 2022 codigo_eleicao = 544 | 2º turno em 2022 codigo_eleicao = 545)

codigo_eleicao = 6257 # para segundo turno, ver o código que será disponibilizado pelo o TSE

codigo_eleicao_formatado = f"{codigo_eleicao:06d}"

codigo_cargo = "0001"  # Presidente

"""
| Cargo              | Código |

| Presidente         | `0001` |
| Governador         | `0003` |
| Senador            | `0005` |
| Deputado Federal   | `0006` |
| Deputado Estadual  | `0007` |
| Deputado Distrital | `0008` |
| Prefeito           | `0011` |
| Vereador           | `0013` |
"""

# ============================================================
# 1. BUSCAR LISTA OFICIAL DE MUNICÍPIOS
# ============================================================

url_municipios = (
    f"{url_base}/"
    f"{ambiente}/"
    f"{ciclo}/"
    f"{codigo_eleicao}/"
    f"config/"
    f"mun-e{codigo_eleicao_formatado}-cm.json"
)


resposta = requests.get(
    url_municipios,
    timeout=30
)

resposta.raise_for_status()

dados_municipios = resposta.json()


# ============================================================
# 2. CRIAR LISTA COM TODOS OS MUNICÍPIOS
# ============================================================

municipios = []

for estado in dados_municipios["abr"]:

    uf = estado["cd"].lower()

    nome_estado = estado["ds"]

    # Ignora localidades do exterior
    if uf == "zz":
        continue

    for municipio in estado["mu"]:

        codigo_tse = str(
            municipio["cd"]
        ).zfill(5)

        codigo_ibge = str(
            municipio.get("cdi", "")
        ).zfill(5)

        nome_municipio = municipio["nm"]

        municipios.append({

            "uf": uf,

            "estado": nome_estado,

            "codigo_tse": codigo_tse,

            "codigo_ibge": codigo_ibge,

            "municipio": nome_municipio

        })


print(
    "Quantidade de municípios encontrados:",
    len(municipios)
)


# ============================================================
# 3. LISTA QUE RECEBERÁ TODOS OS RESULTADOS
# ============================================================

resultados = []


# ============================================================
# 4. PERCORRER TODOS OS MUNICÍPIOS DO BRASIL
# ============================================================

for indice, municipio in enumerate(municipios, start=1):

    uf = municipio["uf"]

    estado = municipio["estado"]

    codigo_tse = municipio["codigo_tse"]

    codigo_ibge = municipio["codigo_ibge"]

    nome_municipio = municipio["municipio"]


    # --------------------------------------------------------
    # URL DO RESULTADO PRESIDENCIAL DO MUNICÍPIO
    # --------------------------------------------------------

    url_resultado = (
        f"{url_base}/"
        f"{ambiente}/"
        f"{ciclo}/"
        f"{codigo_eleicao}/"
        f"dados/{uf}/"
        f"{uf}{codigo_tse}"
        f"-c{codigo_cargo}"
        f"-e{codigo_eleicao_formatado}"
        f"-u.json"
    )


    try:

        resposta = requests.get(
            url_resultado,
            timeout=30
        )


        # Caso o arquivo ainda não esteja disponível
        if resposta.status_code == 404:

            print(
                f"Arquivo não encontrado: "
                f"{nome_municipio} - {uf.upper()}"
            )

            continue


        resposta.raise_for_status()

        dados = resposta.json()


        # ====================================================
        # 5. DADOS GERAIS DA VOTAÇÃO DO MUNICÍPIO
        # ====================================================

        votos = dados.get(
            "v",
            {}
        )

        eleitores = dados.get(
            "e",
            {}
        )

        secoes = dados.get(
            "s",
            {}
        )


        # ----------------------------------------------------
        # VOTOS
        # ----------------------------------------------------

        total_votos = votos.get(
            "tv",
            0
        )

        votos_validos = votos.get(
            "vv",
            0
        )

        votos_brancos = votos.get(
            "vb",
            0
        )

        total_nulos = votos.get(
            "tvn",
            0
        )

        votos_nulos = votos.get(
            "vn",
            0
        )

        votos_nulos_tecnicos = votos.get(
            "vnt",
            0
        )

        votos_anulados = votos.get(
            "van",
            0
        )

        votos_anulados_sub_judice = votos.get(
            "vansj",
            0
        )

        votos_sem_candidato = votos.get(
            "vscv",
            0
        )

        votos_secoes_anuladas = votos.get(
            "vsan",
            0
        )


        # ----------------------------------------------------
        # CRIAMOS UMA COLUNA ANALÍTICA DE "INVÁLIDOS"
        #
        # Não existe no TSE um campo chamado "invalidos".
        #
        # Aqui estamos considerando:
        #
        # nulos
        # + anulados
        # + anulados sub judice
        #
        # O total_nulos já contém:
        # votos_nulos + votos_nulos_tecnicos
        # ----------------------------------------------------

        votos_invalidos = (
            total_nulos
            + votos_anulados
            + votos_anulados_sub_judice
        )


        # ----------------------------------------------------
        # BRANCOS + INVÁLIDOS
        # ----------------------------------------------------

        votos_nao_validos = (
            votos_brancos
            + votos_invalidos
        )


        # ====================================================
        # 6. DADOS DOS ELEITORES
        # ====================================================

        eleitorado = eleitores.get(
            "te",
            0
        )

        comparecimento = eleitores.get(
            "c",
            0
        )

        abstencoes = eleitores.get(
            "a",
            0
        )


        # ====================================================
        # 7. DADOS DAS SEÇÕES
        # ====================================================

        total_secoes = secoes.get(
            "ts",
            0
        )

        secoes_totalizadas = secoes.get(
            "st",
            0
        )

        secoes_nao_totalizadas = secoes.get(
            "snt",
            0
        )

        percentual_secoes_totalizadas = secoes.get(
            "pstn",
            0
        )


        # ====================================================
        # 8. PERCORRER OS CANDIDATOS
        # ====================================================

        for cargo in dados.get("carg", []):

            for agrupamento in cargo.get("agr", []):

                for partido in agrupamento.get("par", []):

                    for candidato in partido.get("cand", []):


                        # ====================================
                        # 9. UMA LINHA DO DATAFRAME
                        # ====================================

                        resultados.append({

                            # -------------------------------
                            # LOCALIZAÇÃO
                            # -------------------------------

                            "estado": estado,

                            "uf": uf.upper(),

                            "municipio": nome_municipio,

                            "codigo_tse": codigo_tse,

                            "codigo_ibge": codigo_ibge,


                            # -------------------------------
                            # CANDIDATO
                            # -------------------------------

                            "numero_candidato":
                                candidato.get("n"),

                            "nome_candidato":
                                candidato.get("nm"),

                            "nome_urna":
                                candidato.get("nmu"),

                            "partido":
                                partido.get("sg"),

                            "destinacao_voto":
                                candidato.get("dvt"),

                            "situacao":
                                candidato.get("st"),

                            "eleito":
                                candidato.get("e"),


                            # -------------------------------
                            # VOTAÇÃO DO CANDIDATO
                            # -------------------------------

                            "votos_candidato":
                                candidato.get("vap", 0),

                            "percentual_candidato":
                                candidato.get("pvap"),

                            "percentual_candidato_preciso":
                                candidato.get("pvapn"),


                            # -------------------------------
                            # TOTALIZAÇÃO DO MUNICÍPIO
                            # -------------------------------

                            "total_votos":
                                total_votos,

                            "votos_validos":
                                votos_validos,

                            "votos_brancos":
                                votos_brancos,

                            "total_votos_nulos":
                                total_nulos,

                            "votos_nulos":
                                votos_nulos,

                            "votos_nulos_tecnicos":
                                votos_nulos_tecnicos,

                            "votos_anulados":
                                votos_anulados,

                            "votos_anulados_sub_judice":
                                votos_anulados_sub_judice,

                            "votos_invalidos":
                                votos_invalidos,

                            "votos_nao_validos":
                                votos_nao_validos,

                            "votos_sem_candidato":
                                votos_sem_candidato,

                            "votos_secoes_anuladas":
                                votos_secoes_anuladas,


                            # -------------------------------
                            # ELEITORADO
                            # -------------------------------

                            "eleitorado":
                                eleitorado,

                            "comparecimento":
                                comparecimento,

                            "abstencoes":
                                abstencoes,


                            # -------------------------------
                            # SEÇÕES
                            # -------------------------------

                            "total_secoes":
                                total_secoes,

                            "secoes_totalizadas":
                                secoes_totalizadas,

                            "secoes_nao_totalizadas":
                                secoes_nao_totalizadas,

                            "percentual_secoes_totalizadas":
                                percentual_secoes_totalizadas,


                            # -------------------------------
                            # CONTROLE DA TOTALIZAÇÃO
                            # -------------------------------

                            "andamento":
                                dados.get("and"),

                            "totalizacao_final":
                                dados.get("tf"),

                            "data_totalizacao":
                                dados.get("dt"),

                            "hora_totalizacao":
                                dados.get("ht"),

                            "data_geracao":
                                dados.get("dg"),

                            "hora_geracao":
                                dados.get("hg")

                        })


        # ====================================================
        # MOSTRAR PROGRESSO
        # ====================================================

        if indice % 100 == 0:

            print(
                f"{indice} de "
                f"{len(municipios)} "
                f"municípios processados"
            )


        # ====================================================
        # PEQUENA PAUSA ENTRE REQUISIÇÕES
        # ====================================================

        time.sleep(0.03)


    except requests.RequestException as erro:

        print(
            f"Erro em "
            f"{nome_municipio} - "
            f"{uf.upper()}: "
            f"{erro}"
        )


# ============================================================
# 10. CRIAR UM ÚNICO DATAFRAME
# ============================================================

df_eleicao = pd.DataFrame(
    resultados
)


# ============================================================
# 11. ORGANIZAR O DATAFRAME
# ============================================================

df_eleicao = df_eleicao.sort_values(
    by=[
        "uf",
        "municipio",
        "votos_candidato"
    ],
    ascending=[
        True,
        True,
        False
    ]
).reset_index(
    drop=True
)


# ============================================================
# 12. RESULTADO
# ============================================================

print()

print(
    "Quantidade de linhas:",
    len(df_eleicao)
)

print()

print(
    "Quantidade de municípios:",
    df_eleicao["municipio"].nunique()
)

print()

print(
    "Quantidade de candidatos:",
    df_eleicao["nome_urna"].nunique()
)


df_eleicao